# Sandbox

In [ ]:
import pandas as pd 
import sys
import os

from importlib import reload
from IPython.display import display

# Asegurar que el path apunte a la raíz para encontrar el paquete 'src'
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.utils import select_file, select_directory

In [ ]:
import numpy as np
import pandas as pd
import numpy_financial as npf

from sqlalchemy import func
from src.database import SessionLocal, Credito, Cuota, Cobranza, Cliente


db = SessionLocal()
cuil = 27446637342
nuevo_tope = 0.25
fecha = pd.Period.now("D")

cuil = str(cuil)
sueldo = float((db.query(Cliente.remuneracion)
          .filter(Cliente.cuil == cuil)
          .scalar()))
print(f"Sueldo: $ {sueldo:,.2f}")

# 1. Crear una subconsulta que agrupe y sume las cobranzas por cada cuota
cobranzas_sumadas = (
    db.query(
        Cobranza.cuota_id,
        func.sum(Cobranza.capital).label("sum_capital"),
        func.sum(Cobranza.interes).label("sum_interes"),
        func.sum(Cobranza.iva).label("sum_iva")
    )
    .group_by(Cobranza.cuota_id)
    .subquery()
)
# 2. Hacer la consulta principal uniendo la subconsulta
query = (
    db.query(
        Cuota.credito_id.label("Crédito"),
        Cuota.nro_cuota.label("Cuota"),
        Cuota.fecha_vencimiento.label("Vencimiento"),
        Credito.tna_c_iva.label("TNA c/IVA"),
        # Usamos .c. para acceder a las columnas de la subconsulta
        (Cuota.capital - func.coalesce(cobranzas_sumadas.c.sum_capital, 0)).label("Capital"),
        (Cuota.interes - func.coalesce(cobranzas_sumadas.c.sum_interes, 0)).label("Interés"),
        (Cuota.iva - func.coalesce(cobranzas_sumadas.c.sum_iva, 0)).label("IVA")
    )
    .join(Credito, Credito.id == Cuota.credito_id)
    .join(Cliente, Cliente.cuil == Credito.cliente_cuil)
    # Hacemos el outerjoin con la subconsulta, no con la tabla directa
    .outerjoin(cobranzas_sumadas, cobranzas_sumadas.c.cuota_id == Cuota.id)
    .filter(Cliente.cuil == str(cuil))
)


df = pd.read_sql(query.statement, db.get_bind())
df.sort_values(by=["Cuota"], inplace=True)
df["Total"] = df[["Capital", "Interés", "IVA"]].sum(axis=1)
df["Vencimiento"] = pd.to_datetime(df["Vencimiento"])
creditos = df.loc[df["Total"].round(0) > 0].groupby("Crédito")[["Cuota", "Total", "TNA c/IVA"]].max()
cuota_max = sueldo * nuevo_tope
filtro = df["Vencimiento"].dt.to_period("M") > pd.Period(fecha, freq="M")
df.loc[filtro, ["Interés", "IVA"]] = .0, .0
df["Total"] = df[["Capital", "Interés", "IVA"]].sum(axis=1)

creditos["TEM c/IVA"] = creditos["TNA c/IVA"] * 30 / 365

print(f"Valor Cuota:\n{creditos["Total"].map("$ {:,.2f}".format)}")
print(f"Cuota Descontable: $ {cuota_max:,.2f}")

if creditos["Total"].sum() <= cuota_max:
    print("✅ Cuota cobrable.")
else:
    for crt, row in creditos.iterrows():
        tem  = row["TNA c/IVA"] * 30 / 365
        nuevo_cap = df.loc[df["Crédito"] == crt, "Total"].sum()
        val_log = (cuota_max - nuevo_cap*tem)/cuota_max
        if val_log <= 0:
            print(f"❌ El crédito nro. {crt} NO se puede refinanciar.\n  🟡 Debe: $ {nuevo_cap:,.2f}\n  🟡 Plazo Original: {int(row["Cuota"])} cuotas")
            print(f"Según la tasa y el capital la cuota más chica que se puede cobrar es de: $ {nuevo_cap * tem:,.2f}")
        else:
            print(f"☑️ Crédito nro. {crt} refinanciables.")
            plazo = int(np.round(-np.log(val_log)/np.log(1+tem)))


df["Vencimiento"] = df["Vencimiento"].dt.to_period("D")
df.set_index(["Crédito", "Cuota", "Vencimiento"], inplace=True)
df.sort_index(inplace=True)
df.loc[("","","Total"), ["Capital", "Interés", "IVA", "Total"]] = df[["Capital", "Interés", "IVA", "Total"]].sum()
df[["Capital", "Interés", "IVA", "Total"]].map("${:,.2f}".format)